In [ ]:
pip install pandas numpy scikit-learn matplotlib

#### **STEP 1: DATASET LOADING AND SPLITTING**

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

In [4]:
# PHASE A: LOAD DATASET & DEFINE TARGET
print("Loading 'student_dropout_dataset_v3.csv'...")
df = pd.read_csv('student_dropout_dataset_v3.csv')
df.head()

Loading 'student_dropout_dataset_v3.csv'...


,Student_ID,Age,Gender,Family_Income,Internet_Access,Study_Hours_per_Day,Attendance_Rate,Assignment_Delay_Days,Travel_Time_Minutes,Part_Time_Job,Scholarship,Stress_Index,GPA,Semester_GPA,CGPA,Semester,Department,Parental_Education,Dropout
0,1,22.1,Male,25000.0,Yes,3.36,86.1,2,20.4,Yes,No,5.5,0.96,0.90,0.90,Year 1,Arts,High School,0
1,2,20.7,Male,25000.0,Yes,4.30,68.0,2,44.0,No,No,6.8,1.28,1.20,1.19,Year 3,Engineering,Bachelor,1
2,3,22.4,Male,40183.0,Yes,4.40,70.9,0,48.9,Yes,No,5.5,1.68,1.32,1.32,Year 1,Arts,Master,0
3,4,24.4,Male,NaN,Yes,NaN,82.2,2,38.6,No,No,NaN,1.78,1.77,1.77,Year 1,CS,High School,1
4,5,20.5,Female,25319.0,Yes,4.19,75.7,1,23.0,No,No,7.0,1.48,0.91,0.87,Year 4,Business,Bachelor,0


In [10]:
# Define X (features) and y (target). Drop 'Student_ID' as it's not a feature.
X = df.drop(['Student_ID'], axis=1)
y = df['Dropout']
X.head()

,Age,Gender,Family_Income,Internet_Access,Study_Hours_per_Day,Attendance_Rate,Assignment_Delay_Days,Travel_Time_Minutes,Part_Time_Job,Scholarship,Stress_Index,GPA,Semester_GPA,CGPA,Semester,Department,Parental_Education,Dropout
0,22.1,Male,25000.0,Yes,3.36,86.1,2,20.4,Yes,No,5.5,0.96,0.90,0.90,Year 1,Arts,High School,0
1,20.7,Male,25000.0,Yes,4.30,68.0,2,44.0,No,No,6.8,1.28,1.20,1.19,Year 3,Engineering,Bachelor,1
2,22.4,Male,40183.0,Yes,4.40,70.9,0,48.9,Yes,No,5.5,1.68,1.32,1.32,Year 1,Arts,Master,0
3,24.4,Male,NaN,Yes,NaN,82.2,2,38.6,No,No,NaN,1.78,1.77,1.77,Year 1,CS,High School,1
4,20.5,Female,25319.0,Yes,4.19,75.7,1,23.0,No,No,7.0,1.48,0.91,0.87,Year 4,Business,Bachelor,0


In [7]:
# PHASE B: DATA CLEANING & ENCODING
print("\nStarting Data Cleaning and Encoding...")

numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X.select_dtypes(include=['object']).columns

numeric_imputer = SimpleImputer(strategy='median')
categorical_imputer = SimpleImputer(strategy='most_frequent')
categorical_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_imputer, numeric_cols),
        ('cat', categorical_encoder, categorical_cols)
    ]
)

X_processed = preprocessor.fit_transform(X)
cat_names = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_cols)
feature_names = np.concatenate([numeric_cols, cat_names])
X_clean = pd.DataFrame(X_processed, columns=feature_names)
X_clean.head()


Starting Data Cleaning and Encoding...


,Age,Family_Income,Study_Hours_per_Day,Attendance_Rate,Assignment_Delay_Days,Travel_Time_Minutes,Stress_Index,GPA,Semester_GPA,CGPA,...,Department_Arts,Department_Business,Department_CS,Department_Engineering,Department_Science,Parental_Education_Bachelor,Parental_Education_High School,Parental_Education_Master,Parental_Education_PhD,Parental_Education_nan
0,22.1,25000.0,3.36,86.1,2.0,20.4,5.5,0.96,0.90,0.90,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1,20.7,25000.0,4.30,68.0,2.0,44.0,6.8,1.28,1.20,1.19,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
2,22.4,40183.0,4.40,70.9,0.0,48.9,5.5,1.68,1.32,1.32,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,24.4,29740.5,4.00,82.2,2.0,38.6,5.5,1.78,1.77,1.77,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,20.5,25319.0,4.19,75.7,1.0,23.0,7.0,1.48,0.91,0.87,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


In [ ]:
# PHASE C: STRATIFIED SPLITTING
print("\nPerforming Stratified Splitting...")

# Split 1: 40% Main Model (Set-1), 60% remainder
X_set1, X_remainder, y_set1, y_remainder = train_test_split(
    X_clean, y, test_size=0.60, stratify=y, random_state=42
)

# Split 2: Divide remainder into Campus 1 and Campus 2
X_set2, X_set3, y_set2, y_set3 = train_test_split(
    X_remainder, y_remainder, test_size=0.50, stratify=y_remainder, random_state=42
)

# Split 3: Create Train/Test splits (80/20) for each Set
X_set1_train, X_set1_test, y_set1_train, y_set1_test = train_test_split(X_set1, y_set1, test_size=0.20, stratify=y_set1, random_state=42)
X_set2_train, X_set2_test, y_set2_train, y_set2_test = train_test_split(X_set2, y_set2, test_size=0.20, stratify=y_set2, random_state=42)
X_set3_train, X_set3_test, y_set3_train, y_set3_test = train_test_split(X_set3, y_set3, test_size=0.20, stratify=y_set3, random_state=42)


Performing Stratified Splitting...


In [ ]:
# PHASE D: DOCUMENTATION & EXPORT
import pandas as pd
from IPython.display import display

print("\n" + "="*25)
print(" DATA SPLIT SUMMARY")
print("="*25)
print(f"Total Original Dataset: {len(X_clean)} rows\n")

split_summary = {
    'Environment': ['Set-1 (Main Model)', 'Set-2 (Campus 1)', 'Set-3 (Campus 2)'],
    'Total Rows': [len(X_set1), len(X_set2), len(X_set3)],
    'Training Rows (80%)': [len(X_set1_train), len(X_set2_train), len(X_set3_train)],
    'Testing Rows (20%)': [len(X_set1_test), len(X_set2_test), len(X_set3_test)]
}
df_splits = pd.DataFrame(split_summary)

print("--- Data Split Summary ---")
display(df_splits) 

print("\n")

strat_summary = {
    'Environment': ['Set-1 (Main Model)', 'Set-2 (Campus 1)', 'Set-3 (Campus 2)'],
    'No Dropout (0)': [
        f"{y_set1.value_counts(normalize=True).get(0, 0) * 100:.2f}%",
        f"{y_set2.value_counts(normalize=True).get(0, 0) * 100:.2f}%",
        f"{y_set3.value_counts(normalize=True).get(0, 0) * 100:.2f}%"
    ],
    'Dropout (1)': [
        f"{y_set1.value_counts(normalize=True).get(1, 0) * 100:.2f}%",
        f"{y_set2.value_counts(normalize=True).get(1, 0) * 100:.2f}%",
        f"{y_set3.value_counts(normalize=True).get(1, 0) * 100:.2f}%"
    ]
}
df_strat = pd.DataFrame(strat_summary)

print("--- Stratification Check (Dropout Distribution) ---")
display(df_strat)

print("="*45)

# Save Set-1 (Main Server)
X_set1_train.assign(Dropout=y_set1_train).to_csv('set1_train.csv', index=False)
X_set1_test.assign(Dropout=y_set1_test).to_csv('set1_test.csv', index=False)

# Save Set-2 (Campus 1)
X_set2_train.assign(Dropout=y_set2_train).to_csv('set2_train.csv', index=False)
X_set2_test.assign(Dropout=y_set2_test).to_csv('set2_test.csv', index=False)

# Save Set-3 (Campus 2)
X_set3_train.assign(Dropout=y_set3_train).to_csv('set3_train.csv', index=False)
X_set3_test.assign(Dropout=y_set3_test).to_csv('set3_test.csv', index=False)

print("\n6 CSV files generated.")


 DATA SPLIT SUMMARY
Total Original Dataset: 10000 rows

--- Data Split Summary ---


,Environment,Total Rows,Training Rows (80%),Testing Rows (20%)
0,Set-1 (Main Model),4000,3200,800
1,Set-2 (Campus 1),3000,2400,600
2,Set-3 (Campus 2),3000,2400,600




--- Stratification Check (Dropout Distribution) ---


,Environment,No Dropout (0),Dropout (1)
0,Set-1 (Main Model),76.45%,23.55%
1,Set-2 (Campus 1),76.47%,23.53%
2,Set-3 (Campus 2),76.47%,23.53%



6 CSV files generated.


#### **STEP 2 : DEVELOP THE MAIN MODEL**

In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import joblib
import os

print(" STEP 2: DEVELOP THE MAIN MODEL")

# 1. UPDATED TO USE YOUR NEW FILE NAMES
print("Loading Global Train and Test data...")
train_data = pd.read_csv('global_train.csv') 
test_data = pd.read_csv('global_test.csv')

# Separate features (X) and target (y) for Training
X_train = train_data.drop('Dropout', axis=1)
y_train = train_data['Dropout']

# Separate features (X) and target (y) for Testing
X_test = test_data.drop('Dropout', axis=1)
y_test = test_data['Dropout']

# Train the model using GLOBAL TRAINING DATA
print("Training Random Forest Classifier on Global Training Data...")
main_model = RandomForestClassifier(n_estimators=100, random_state=42)
main_model.fit(X_train, y_train)

# Evaluate using GLOBAL TEST DATA
print("Evaluating model on Global Test Data...\n")
y_pred = main_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print("--- MAIN MODEL EVALUATION METRICS ---")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

# 2. SAVE THE MODEL
model_filename = 'main_model.pkl'
joblib.dump(main_model, model_filename)
print(f"\nSUCCESS: Model saved successfully as '{model_filename}'")

 STEP 2: DEVELOP THE MAIN MODEL
Loading Global Train and Test data...
Training Random Forest Classifier on Global Training Data...
Evaluating model on Global Test Data...

--- MAIN MODEL EVALUATION METRICS ---
Accuracy:  0.8063
Precision: 0.6486
Recall:    0.3830
F1 Score:  0.4816

SUCCESS: Model saved successfully as 'main_model.pkl'


#### **STEP 4 : DEPLOY THE MAIN MODEL**

In [3]:
import pandas as pd
import joblib

print(" PREPARING CAMPUS ENVIRONMENTS (SPACE 1 & 2)")

# "Download" the Main Model from the Cloud to the Campuses
print("Downloading 'main_model.pkl' from Central Server...")
campus1_model = joblib.load('Cloud-1_Main_Server/main_model.pkl')
campus2_model = joblib.load('Cloud-1_Main_Server/main_model.pkl')

# Load Campus 1 Local Data (Space-1)
print("Loading Campus 1 Data...")
c1_train = pd.read_csv('Cloud-2_Campuses/Space-1_Campus-1/set2_train.csv')
X_campus1_train = c1_train.drop('Dropout', axis=1)
y_campus1_train = c1_train['Dropout']

c1_test = pd.read_csv('Cloud-2_Campuses/Space-1_Campus-1/set2_test.csv')
X_campus1_test = c1_test.drop('Dropout', axis=1)
y_campus1_test = c1_test['Dropout']

# Load Campus 2 Local Data (Space-2)
print("Loading Campus 2 Data...")
c2_train = pd.read_csv('Cloud-2_Campuses/Space-2_Campus-2/set3_train.csv')
X_campus2_train = c2_train.drop('Dropout', axis=1)
y_campus2_train = c2_train['Dropout']

c2_test = pd.read_csv('Cloud-2_Campuses/Space-2_Campus-2/set3_test.csv')
X_campus2_test = c2_test.drop('Dropout', axis=1)
y_campus2_test = c2_test['Dropout']

print("\nSUCCESS: Main model successfully deployed to Campus 1 and Campus 2!")
print("Both campuses have their local data loaded and are ready for instructions.")

 PREPARING CAMPUS ENVIRONMENTS (SPACE 1 & 2)
Loading Campus 1 Data...
Loading Campus 2 Data...

SUCCESS: Main model successfully deployed to Campus 1 and Campus 2!
Both campuses have their local data loaded and are ready for instructions.


#### **STEP 5 : CAMPUS-1 EVALUATION**

In [4]:
from sklearn.metrics import accuracy_score
import joblib

print(" STEP 5: CAMPUS-1 EVALUATION")

baseline_accuracy = 0.8063 

print(f"Baseline Main Model Accuracy: {baseline_accuracy:.4f}\n")

#  INITIAL TEST (50 Samples)
print("--- INITIAL TEST ---")
X_test_50 = X_campus1_test.iloc[:50]
y_test_50 = y_campus1_test.iloc[:50]

initial_preds = campus1_model.predict(X_test_50)
initial_acc = accuracy_score(y_test_50, initial_preds)

print(f"Tested on 50 local Set-2 samples.")
print(f"Initial Local Accuracy: {initial_acc:.4f}\n")

# EVALUATE SCENARIOS
if initial_acc < baseline_accuracy:
    print("--- SCENARIO A TRIGGERED: Accuracy Reduced ---")
    print(f"Because {initial_acc:.4f} is lower than {baseline_accuracy:.4f}, we proceed directly to retraining.\n")
    
else:
    print("--- SCENARIO B TRIGGERED: Accuracy Not Reduced ---")
    print(f"Because {initial_acc:.4f} is >= {baseline_accuracy:.4f}, testing additional 20 samples...")
    
    # Test on 70 samples total
    X_test_70 = X_campus1_test.iloc[:70]
    y_test_70 = y_campus1_test.iloc[:70]
    
    preds_70 = campus1_model.predict(X_test_70)
    acc_70 = accuracy_score(y_test_70, preds_70)
    
    print(f"Accuracy on 70 samples: {acc_70:.4f}")
    
    if acc_70 < baseline_accuracy:
        print("Result: Accuracy degraded after additional testing. Proceeding to retrain.\n")
    else:
        print("Result: Accuracy did not degrade. Proceeding to retrain anyway to optimize locally.\n")

# RETRAIN, TEST ALL, AND SAVE
print("--- RETRAINING MODEL ---")
print("Retraining model using Set-2 training data...")
campus1_model.fit(X_campus1_train, y_campus1_train)

print("Testing the retrained model using ALL Set-2 test samples...")
final_c1_preds = campus1_model.predict(X_campus1_test)
final_c1_acc = accuracy_score(y_campus1_test, final_c1_preds)

print(f"Final Retrained Campus-1 Accuracy: {final_c1_acc:.4f}\n")

# Save the model
c1_model_name = 'campus_1_v2.pkl'
joblib.dump(campus1_model, c1_model_name)
print(f"SUCCESS: Model saved as '{c1_model_name}'")

 STEP 5: CAMPUS-1 EVALUATION
Baseline Main Model Accuracy: 0.8063

--- INITIAL TEST ---
Tested on 50 local Set-2 samples.
Initial Local Accuracy: 0.6800

--- SCENARIO A TRIGGERED: Accuracy Reduced ---
Because 0.6800 is lower than 0.8063, we proceed directly to retraining.

--- RETRAINING MODEL ---
Retraining model using Set-2 training data...
Testing the retrained model using ALL Set-2 test samples...
Final Retrained Campus-1 Accuracy: 0.8017

SUCCESS: Model saved as 'campus_1_v2.pkl'


#### **STEP 6 : CAMPUS-2 EVALUATION**

In [6]:
from sklearn.metrics import accuracy_score
import joblib

print(" STEP 6: CAMPUS-2 EVALUATION")
baseline_accuracy = 0.8063 

# INITIAL TEST (75 Samples)
print("--- INITIAL TEST ---")
# Slice exactly 75 samples as requested by the rubric
X_test_75 = X_campus2_test.iloc[:75]
y_test_75 = y_campus2_test.iloc[:75]

initial_preds_2 = campus2_model.predict(X_test_75)
initial_acc_2 = accuracy_score(y_test_75, initial_preds_2)

print(f"Tested on 75 local Set-3 samples.")
print(f"Initial Local Accuracy: {initial_acc_2:.4f}\n")

# 2. EVALUATE SCENARIOS (C & D)
if initial_acc_2 < baseline_accuracy:
    print("--- SCENARIO C TRIGGERED: Accuracy Reduced ---")
    print(f"Because {initial_acc_2:.4f} is lower than {baseline_accuracy:.4f}, proceeding directly to retraining.\n")
    
else:
    print("--- SCENARIO D TRIGGERED: Accuracy Not Reduced ---")
    print(f"Because {initial_acc_2:.4f} is >= {baseline_accuracy:.4f}, testing an additional 20 samples...")
    
    # Test on 95 samples total
    X_test_95 = X_campus2_test.iloc[:95]
    y_test_95 = y_campus2_test.iloc[:95]
    
    preds_95 = campus2_model.predict(X_test_95)
    acc_95 = accuracy_score(y_test_95, preds_95)
    
    print(f"Accuracy on 95 samples: {acc_95:.4f}")
    
    if acc_95 < baseline_accuracy:
        print("Result: Accuracy degraded after additional testing. Proceeding to retrain.\n")
    else:
        print("Result: Accuracy did not degrade. Proceeding to retrain anyway to optimize locally.\n")


# RETRAIN, TEST 125 SAMPLES, AND SAVE

print("--- RETRAINING MODEL ---")
print("Retraining model using Set-3 training data...")
campus2_model.fit(X_campus2_train, y_campus2_train)

print("Testing the retrained model using exactly 125 Set-3 test samples...")
# Slice exactly 125 samples for the final test
X_test_125 = X_campus2_test.iloc[:125]
y_test_125 = y_campus2_test.iloc[:125]

final_c2_preds = campus2_model.predict(X_test_125)
final_c2_acc = accuracy_score(y_test_125, final_c2_preds)

print(f"Final Retrained Campus-2 Accuracy: {final_c2_acc:.4f}\n")

c2_model_name = 'campus_2_v2.pkl'
joblib.dump(campus2_model, c2_model_name)
print(f"SUCCESS: Model saved as '{c2_model_name}'")

 STEP 6: CAMPUS-2 EVALUATION
--- INITIAL TEST ---
Tested on 75 local Set-3 samples.
Initial Local Accuracy: 0.7867

--- SCENARIO C TRIGGERED: Accuracy Reduced ---
Because 0.7867 is lower than 0.8063, proceeding directly to retraining.

--- RETRAINING MODEL ---
Retraining model using Set-3 training data...
Testing the retrained model using exactly 125 Set-3 test samples...
Final Retrained Campus-2 Accuracy: 0.7920

SUCCESS: Model saved as 'campus_2_v2.pkl'


#### **STEP 7 : RETRIEVE MODELS VIA API (SIMULATION)**

In [7]:
import shutil
import os

print(" STEP 7: RETRIEVE MODELS VIA API (SIMULATION)")

print("Initiating API GET requests from Main Server...\n")

def simulate_api_download(endpoint, filename):
    print(f"--> [GET] {endpoint}")
    # In a real Flask app, this would be: 
    # response = requests.get(f'http://localhost:{port}{endpoint}')
    # open(filename, 'wb').write(response.content)
    
    # For local notebook simulation, we verify the file exists locally
    if os.path.exists(filename):
        print(f"    SUCCESS: 200 OK. '{filename}' retrieved and stored locally in Central Environment.\n")
        return True
    else:
        print(f"    ERROR: 404. '{filename}' not found.\n")
        return False

# Simulate retrieving from Campus 1 (Hospital 1)
simulate_api_download('/campus1/model', 'campus_1_v2.pkl')

# Simulate retrieving from Campus 2 (Hospital 2)
simulate_api_download('/campus2/model', 'campus_2_v2.pkl')

print("Step 7 Complete: Both v2 models have been successfully retrieved to the central environment.")

 STEP 7: RETRIEVE MODELS VIA API (SIMULATION)
Initiating API GET requests from Main Server...

--> [GET] /campus1/model
    SUCCESS: 200 OK. 'campus_1_v2.pkl' retrieved and stored locally in Central Environment.

--> [GET] /campus2/model
    SUCCESS: 200 OK. 'campus_2_v2.pkl' retrieved and stored locally in Central Environment.

Step 7 Complete: Both v2 models have been successfully retrieved to the central environment.


#### **STEP 8 : MODEL AGGREGATION**

In [10]:
import joblib
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


print("STEP 8: FEDERATED AGGREGATION")

# Load the locally retrained models
print("Loading v2 models from Campus 1 and Campus 2...")
model_c1 = joblib.load('campus_1_v2.pkl')
model_c2 = joblib.load('campus_2_v2.pkl')

# Combine the Random Forests 
print("Aggregating models (Merging the Random Forests)...")
combined_estimators = model_c1.estimators_ + model_c2.estimators_

# Load the original main model to act as our base template
# (Using the path from your Cloud-1 folder)
global_model_v2 = joblib.load('Cloud-1_Main_Server/main_model.pkl')

# Overwrite the base model's trees with our new massive combined forest
global_model_v2.estimators_ = combined_estimators
global_model_v2.n_estimators = len(combined_estimators)

print("Loading Central Server Set-1 Test Data...")
set1_test = pd.read_csv('Cloud-1_Main_Server/set1_test.csv')
X_test = set1_test.drop('Dropout', axis=1)
y_test = set1_test['Dropout']

# Evaluate the New Global Model on the ORIGINAL Set-1 Test Data
print("\nEvaluating the Aggregated Global Model on Set-1 Test Data...")
y_pred_global = global_model_v2.predict(X_test)

new_acc = accuracy_score(y_test, y_pred_global)
new_prec = precision_score(y_test, y_pred_global, zero_division=0)
new_rec = recall_score(y_test, y_pred_global, zero_division=0)
new_f1 = f1_score(y_test, y_pred_global, zero_division=0)

print("\n--- NEW GLOBAL MODEL METRICS ---")
print(f"Accuracy:  {new_acc:.4f}")
print(f"Precision: {new_prec:.4f}")
print(f"Recall:    {new_rec:.4f}")
print(f"F1 Score:  {new_f1:.4f}")

joblib.dump(global_model_v2, 'main_model_v2.pkl')
print("\nSUCCESS: Aggregated model saved as 'main_model_v2.pkl'")

STEP 8: FEDERATED AGGREGATION
Loading v2 models from Campus 1 and Campus 2...
Aggregating models (Merging the Random Forests)...
Loading Central Server Set-1 Test Data...

Evaluating the Aggregated Global Model on Set-1 Test Data...

--- NEW GLOBAL MODEL METRICS ---
Accuracy:  0.8113
Precision: 0.6480
Recall:    0.4309
F1 Score:  0.5176

SUCCESS: Aggregated model saved as 'main_model_v2.pkl'
